In [16]:
import pandas as pd
from scripts.adjusted_price_data import get_adjusted_price_of_all_companies, indices_data
from scripts.indicator_calculations import calculate_rsi
from scripts.gemini_output import generate_response, dataframe_to_json
import re 

# Get historical data
indices_history_all = indices_data()

# Data cleaning, convert numeric columns to float, handle non-numeric values like commas or #NA
indices_history_all[["Open", "High", "Low", "Close", "Volume"]] = indices_history_all[["Open", "High", "Low", "Close", "Volume"]].apply(pd.to_numeric, errors='coerce')
indices_history_all.set_index('Date', inplace=True)

# Pivot the DataFrame to have 'Date' as index and 'Ticker' as columns for Close prices
pivot_indices_history = indices_history_all.pivot_table(index="Date", columns="Ticker", values="Close")
pivot_indices_history = pivot_indices_history.sort_values(by="Date", ascending=True)
pivot_indices_history = pivot_indices_history.bfill()  # Backfill missing values if any

# Get the latest available date (last date in the index)
last_date = pivot_indices_history.index[-1]
previous_day = pivot_indices_history.index[-2]

# Calculate daily return for the latest date (percentage change from the previous day)
daily_return = ((pivot_indices_history.loc[last_date] - pivot_indices_history.loc[previous_day]) / pivot_indices_history.loc[previous_day]) * 100

# Calculate 20-day and 50-day moving averages for all sectors at once
ma_20_df = pivot_indices_history.rolling(window=20, min_periods=1).mean()
ma_50_df = pivot_indices_history.rolling(window=50, min_periods=1).mean()

# Get the 20-day and 50-day moving averages for the latest available date
ma_20_latest = ma_20_df.loc[last_date]
ma_50_latest = ma_50_df.loc[last_date]

# Calculate RSI (assuming RSI is already calculated in 'rsi_df' and it has the same index as pivot_indices_history)
rsi_df = calculate_rsi(pivot_indices_history)
rsi_latest = rsi_df.loc[last_date]

# Pivot the data for Volume to get it in the same format
volume_pivot = indices_history_all.pivot_table(index="Date", columns="Ticker", values="Volume")
volume_pivot = volume_pivot.sort_values(by="Date", ascending=True)
volume_pivot = volume_pivot.bfill()  # Backfill missing values if any

# Get the latest Volume for each sector
volume_latest = volume_pivot.loc[last_date]

# Get the latest Close prices for each sector
latest_close = pivot_indices_history.loc[last_date]

# Now, create the final summary DataFrame by concatenating all the indicators
summary_df = pd.DataFrame({
    'Volume': volume_latest,
    'Daily Return (%)': daily_return,
    'MA 20': ma_20_latest,
    'MA 50': ma_50_latest,
    'RSI': rsi_latest,
    'LTP': latest_close
})
# Convert the summary DataFrame to JSON
summary_json = dataframe_to_json(summary_df)

# Improved prompt with engaging Nepali greeting, no mention of Gemini
# Assuming you have a function like this for generating responses from the AI
def generate_inspirational_quote():
    # Send prompt to AI model to generate a stock market-related quote in Nepali
    prompt = "Generate an inspirational quote about stock markets and investing in Nepali."
    quote = generate_response(prompt)  # This would call your AI model (e.g., GPT-3, GPT-4)
    return quote

# Now generate the inspirational quote dynamically
today_quote = generate_inspirational_quote()

# Use this dynamically generated quote in your prompt text
prompt_text = f"""
🌞 **नमस्ते, नेपालका लगानीकर्ताहरू!** 🇳🇵🚀

आजको बजारको प्रदर्शनमा गहिरो विश्लेषण गर्न र आजको व्यापारमा लुकेका रत्नहरू र सर्तक संकेतहरू पत्ता लगाउन तयार हुनुहोस्। इन्भेस्टमेन्टका लागि केही महत्वपूर्ण विचारहरू र चाखलाग्दा विश्लेषणहरू ल्याउने छौं, केही इमोजीको साथ! ✨

तपाईंको डेटा यस JSON ढाँचामा प्रस्तुत गरिएको छ:

{summary_json}

**📊 विश्लेषणका प्रमुख बुँदाहरू:**

1. **मुख्य प्रवृत्तिहरू:**
   - कुन क्षेत्रहरू **सशक्त** छन् र कुन **कमजो**र? **एमए 20**, **एमए 50**, **दैनिक प्रतिफल**, र **आरएसआई** का आधारमा क्षेत्रहरूको तुलना गर्नुहोस्।
   - कुन क्षेत्रहरूको **आरएसआई** ७० भन्दा माथि छ र कुन क्षेत्रहरू **आरएसआई ३० भन्दा तल** छन्?
   - आजको **दैनिक प्रतिफल** का आधारमा कुन क्षेत्रहरू **उदाउँछ** (सकारात्मक प्रतिफल) वा **गिरिरहेको** (नकारात्मक प्रतिफल) छन्?

2. **🔍 असामान्य वा महत्त्वपूर्ण निरीक्षणहरू:**
   - के कुनै क्षेत्र **वृद्धि भएको भोल्युम** सँगै **मूल्य परिवर्तन** देखिन्छ?
   - दैनिक प्रतिफल वा महत्वपूर्ण परिवर्तन भएका क्षेत्रहरूमा के **आउटलाइर** छन्?
   - के कुनै **बुलिस डाइवर्जन्स** वा **बेयर ट्रेन्ड** छन् जसले भविष्यको प्रदर्शनलाई असर पुर्याउन सक्छ?

3. **💡 भोल्युम र मूल्यका लागि विचारहरू:**
   - प्रत्येक क्षेत्रको **भोल्युम** परिवर्तन र **मूल्य** चालहरूको सहसंबन्ध विश्लेषण गर्नुहोस्।
   - कुन क्षेत्रहरूमा **भोल्युम बढ्दै** छ, तर **मूल्य घट्दै** छ, जसले **रिभर्सल** वा **संचय** को संकेत दिन सक्छ?

4. **🚨 संभावित जोखिमहरू:**
   - के कुनै क्षेत्रहरू **उच्च जोखिम** को लगानी अवसर प्रस्तुत गर्छन् जसमा अनियमित भोल्युम वा **आरएसआई एक्स्ट्रीम** छन्?
   - कुन क्षेत्रहरूलाई **ध्यान दिनुपर्ने** छ जो सम्भावित **सुधार** वा **मोमेन्टम शिफ्ट** को लागि छन्?

5. **🧠 लगानीकर्ताहरूका लागि सुझावहरू:**
   - आजको बजार डेटा अनुसार, लगानीकर्ताहरूलाई कहाँ आफ्नो ध्यान केन्द्रित गर्नुपर्छ (जस्तै, बलियो आरएसआई भएका बुलिस क्षेत्रहरू, वा सुधारको लागि हेर्नुपर्ने कमजोरा क्षेत्रहरू)?

---

📣 **प्रेरणादायक उद्धरण**:
“{today_quote}”

🚀 प्रवृत्तिहरू हेर्दै, सूचित हुँदै, र बजारको चालसँगै लगानी गर्नुहोस्! 💸📉📈
"""

# Generate the response using Gemini
gemini_output = generate_response(prompt_text)
cleaned_response = re.sub(r'\*\*(.*?)\*\*', r'\1', gemini_output)
print(cleaned_response)

🌞 नमस्ते, नेपालका लगानीकर्ताहरू! 🇳🇵🚀

आजको बजारको प्रदर्शनमा गहिरो विश्लेषणका साथ, यहाँ तपाईंको लागि केही महत्वपूर्ण अन्तर्दृष्टिहरू छन्:

---

📊 आजको बजारको गहिरो विश्लेषण:

१. मुख्य प्रवृत्तिहरू (Main Trends):

*   सशक्त (Strong) क्षेत्रहरू (सकारात्मक प्रतिफल र LTP > MA20 > MA50 को प्रवृत्ति):
    *   HydroPower Index: 🚀 आजको सबैभन्दा बलियो क्षेत्र, सर्वाधिक दैनिक प्रतिफल (+१.५९%) र ठूलो भोल्युमका साथ। यसको RSI (७३.७१) पनि उच्च छ, जसले बलियो मोमेन्टम देखाउँछ।
    *   Banking SubIndex: ✅ राम्रो दैनिक प्रतिफल (+०.८४%) र RSI ७० भन्दा माथि (७१.६७)।
    *   Development Bank Index: 📈 सकारात्मक प्रतिफल (+०.६४%) र RSI ७० भन्दा माथि (७१.९६)।
    *   Manufacturing And Processing: 💪 सकारात्मक प्रतिफल (+०.२९%) र RSI ७० भन्दा माथि (७०.५४)।
    *   Investment: ✨ सकारात्मक प्रतिफल (+०.६१%) र राम्रो RSI (६५.०७)।
    *   Mutual Fund: 💰 सकारात्मक प्रतिफल (+०.३६%) र बलियो RSI (६९.२९)।
    *   Nepse Index: 🇳🇵 समग्र बजार पनि सकारात्मक प्रतिफल (+०.४८%) र बलियो RSI (६९.५५) का साथ बलियो देखिएको छ।

*   कमजो

In [1]:
from momentum_analysis_gemini import gemini_momentum_analysis

print(gemini_momentum_analysis())

🔥 Momentum Market Insight

🏆 नेतृत्वकर्ता स्टकहरू (Leaders):
- SABBL, RSML → ३ हप्तामा क्रमशः २८२% र २४५% को असाधारण वृद्धिसँगै बजारको सबैभन्दा बलियो मोमेन्टम बोकेका स्टकहरू।
- NHPC → १ हप्तादेखि ६ महिनासम्मका लगभग सबै अवधिमा अटाएको सबैभन्दा स्थिर र भरपर्दो 'Trend Holder'।

⚡ तीव्र गति (Acceleration):
- HFIN, SOHL → पछिल्लो १ देखो ४ दिनमा ४६% सम्मको वृद्धि देखिँदै छोटो अवधिमा तीव्र गति समातेका स्टकहरू।
- CORBL → १-दिन देखि १-हप्ताको डेटामा लगातार उपस्थिति जनाउँदै 'Strength' थप्दै गरेको।

❄️ सुस्तता (Cooling):
- AKJCL → ३ देखि ६ महिनाको ऐतिहासिक डेटामा बलियो देखिए पनि पछिल्लो १ हप्तामा यसको माथिल्लो क्रमको उपस्थिति घट्दै गएको।
- TVCL, UNHPL → लामो अवधिमा देखिए पनि छोटो अवधिका शीर्ष सूचिहरूमा नदेखिएका।

🔁 सेक्टर रोटेशन (Rotation):
- बजारमा स्पष्ट रूपमा हाइड्रोपावर (NHPC, AKJCL, SSHL) बाट पैसा झिकिएर माइक्रोफाइनान्स र फाइनान्स (SABBL, RSML, HFIN, CORBL) तिर तीव्र गतिमा मोडिएको देखिन्छ।

📈 ट्रेन्डको गुणस्तर (Trend Quality):
- Consistent (स्थिर): NHPC, SSHL, BHL र RIDI (धेरै समय-अवधिमा निरन